# VK Shelf Intelligence v2.0:  Qwen vs Gemma Multi-Model Demo

**4 models, 2 families, same shelf** — pick one per session (T4 can't hold two simultaneously)

### Model Comparison

| Model | Brand | Year | VRAM | JSON Quality | Speed | Best For |
|-------|-------|------|------|-------------|-------|----------|
| **Qwen2.5-VL-3B** | Alibaba | Jan 2025 | ~6GB | ★★★★☆ | ★★★☆☆ | Reliable structured JSON |
| **Qwen3-VL-2B** | Alibaba | Oct 2025 | ~4GB | ★★★★★ | ★★★★☆ | Best accuracy/size ratio |
| **Gemma 4 E4B** | Google | Apr 2026 | ~4GB | ★★★★☆ | ★★★★★ | Latest, fastest, Google brand |
| **Gemma 3n E4B** | Google | Jun 2025 | ~3GB | ★★★☆☆ | ★★★★★ | Smallest, on-device ready |

**All features that ParallelDots ShelfWatch, Trax, and Zipline charge ₹5–20L/yr for — running free on a T4 GPU.**

### Feature Coverage

| S.No. | Feature | Market Standard | This Notebook |
|---|---------|----------------|---------------|
| 1 | **Reference Product Matching** | Upload SKU → find on shelf | ✅ Multimodal VLM |
| 2 | **Share of Shelf (SoS)** | Brand visibility % | ✅ Per-brand calculation |
| 3 | **On-Shelf Availability (OSA)** | Product present/absent | ✅ Detection + count |
| 4 | **Out-of-Stock Detection** | Empty shelf spots | ✅ Gap identification |
| 5 | **Facing Count** | # of product faces visible | ✅ Per-SKU count |
| 6 | **Stock Level Estimation** | High / Medium / Low | ✅ Per-product rating |
| 7 | **Eye-Level Visibility Analysis** | Premium placement tracking | ✅ Shelf-position scoring |
| 8 | **Customer Visibility Probability** | Statistical attention model | ✅ Position-based probability |
| 9 | **Planogram Compliance** | Reference vs actual comparison | ✅ Two-image comparison |
| 10 | **Competition Analysis** | Your brand vs competitors | ✅ Multi-brand breakdown |
| 11 | **Restock Priority Scoring** | Urgency ranking | ✅ Weighted priority score |
| 12 | **POSM Detection** | Promotional materials visible | ✅ VLM-powered detection |
| 13 | **Price Tag Detection** | Price visibility check | ✅ OCR-capable model |
| 14 | **Shelf Health Score** | Overall shelf condition 0–100 | ✅ Composite scoring |

**Run each model in a SEPARATE session. Restart runtime → change MODEL_CHOICE → run all cells.**

---
*by [@VK_Venkatkumar](https://github.com/vk-ant) — Kaggle Master | M.Tech AI, BITS Pilani*

# Install Necessary Library


In [1]:
!pip3 install -q --upgrade pip
!pip3 install -q "transformers>=4.53.0" accelerate torch torchvision timm gradio qwen-vl-utils

In [2]:


import torch, time, json, re, math, sys, types
from PIL import Image
import warnings; warnings.filterwarnings('ignore')

# Create qwen_vl_utils inline — no separate install needed
def _process_vision_info(messages):
    images = []
    for msg in messages:
        for item in msg.get('content', []):
            if item.get('type') == 'image':
                img = item.get('image')
                if img is not None:
                    images.append(img)
    return images, None

mod = types.ModuleType('qwen_vl_utils')
mod.process_vision_info = _process_vision_info
sys.modules['qwen_vl_utils'] = mod

print(f'GPU: {torch.cuda.get_device_name()}' if torch.cuda.is_available() else 'CPU mode')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

GPU: Tesla T4
VRAM: 15.6 GB


# Pick your model

In [3]:

MODEL_CHOICE = 'qwen2.5-vl-3b'     # RECOMMENDED
# MODEL_CHOICE = 'qwen3-vl-2b'
# MODEL_CHOICE = 'gemma4-e4b'
# MODEL_CHOICE = 'gemma3n-e4b'

print(f'Selected: {MODEL_CHOICE}')

Selected: qwen2.5-vl-3b


# Load model

In [4]:

from transformers import AutoProcessor
t0 = time.time()

QWEN_FAMILY = MODEL_CHOICE.startswith('qwen')

if MODEL_CHOICE == 'qwen3-vl-2b':
    from transformers import Qwen3VLForConditionalGeneration
    MODEL_ID = 'Qwen/Qwen3-VL-2B-Instruct'
    model = Qwen3VLForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto')
elif MODEL_CHOICE == 'qwen2.5-vl-3b':
    from transformers import Qwen2_5_VLForConditionalGeneration
    MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto')
elif MODEL_CHOICE == 'gemma4-e4b':
    from transformers import Gemma4ForConditionalGeneration
    MODEL_ID = 'google/gemma-4-E4B-it'
    model = Gemma4ForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
elif MODEL_CHOICE == 'gemma3n-e4b':
    from transformers import Gemma3nForConditionalGeneration
    MODEL_ID = 'google/gemma-3n-E4B-it'
    model = Gemma3nForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')

processor = AutoProcessor.from_pretrained(MODEL_ID)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'{MODEL_ID} loaded in {time.time()-t0:.1f}s | VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f}GB')

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Qwen/Qwen2.5-VL-3B-Instruct loaded in 45.3s | VRAM: 7.5GB


# Inference engine


In [5]:
from qwen_vl_utils import process_vision_info

def run_vlm(images, prompt, max_tokens=800):
    if not isinstance(images, list): images = [images]
    processed = []
    for img in images:
        if isinstance(img, str): img = Image.open(img)
        img = img.convert('RGB')
        if max(img.size) > 1024:
            r = 1024 / max(img.size)
            img = img.resize((int(img.size[0]*r), int(img.size[1]*r)), Image.LANCZOS)
        processed.append(img)

    content = [{'type': 'image', 'image': img} for img in processed]
    content.append({'type': 'text', 'text': prompt})
    messages = [{'role': 'user', 'content': content}]

    if QWEN_FAMILY:
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                           padding=True, return_tensors='pt').to(DEVICE)
    else:
        inputs = processor.apply_chat_template(
            messages, add_generation_prompt=True,
            tokenize=True, return_dict=True, return_tensors='pt'
        ).to(DEVICE)

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.1, do_sample=True)
    latency = time.time() - t0

    gen_ids = out[0][len(inputs['input_ids'][0]):]
    output = processor.decode(gen_ids, skip_special_tokens=True).strip()

    # Clean thinking tags
    if '<think>' in output:
        idx = output.rfind('</think>')
        if idx != -1:
            output = output[idx + len('</think>'):].strip()
        else:
            start = output.find('{'); end = output.rfind('}')
            if start != -1 and end != -1:
                output = output[start:end+1]
    if '<|channel>' in output:
        idx = output.rfind('<channel|>')
        if idx != -1:
            output = output[idx + len('<channel|>'):].strip()
    output = re.sub(r'<\|.*?\|>', '', output).strip()

    return output, round(latency, 2), len(gen_ids)

print('Engine ready.')

Engine ready.


In [6]:
def clean_output(text):
    """Strip markdown symbols for clean display."""
    text = text.replace('###', '').replace('**', '').replace('* ', '- ')
    text = re.sub(r'^#+\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

# Prompts


In [7]:

PROMPTS = {}

PROMPTS['full_analysis'] = '''You are a retail shelf analyst. Analyze this shelf image.

Report the following clearly:

PRODUCTS DETECTED:
List each unique product with brand name, shelf level (1=top), position (left/center/right), facing count, and stock level (high/medium/low).

BRAND SHARE OF SHELF:
For each brand, estimate what percentage of total shelf space they occupy.

EMPTY SPOTS:
Count and locate any empty gaps on the shelf.

ALERTS:
List top 3 actionable recommendations (restock, rearrange, etc).

SHELF HEALTH SCORE:
Rate overall shelf condition from 0 to 100.

Use simple product names. List each product only once. Be concise.'''

PROMPTS['reference_match'] = '''Image 1 is a REFERENCE PRODUCT photo. Image 2 is a RETAIL SHELF.

Your task: Find ONLY the reference product on the shelf.

Report:
- Product name from the reference image
- Found on shelf: Yes or No
- How many instances found
- Confidence: 0-100%
- Location: which shelf level and position (left/center/right)
- Facing count for each location
- Stock level: high / medium / low
- Share of shelf: what % of total shelf does this product occupy
- Competitors: what products are placed directly next to it
- Recommendation: one actionable insight

IMPORTANT: Report ONLY about the reference product. Do NOT list all products on the shelf.'''

PROMPTS['text_search'] = '''I am looking for "{product_name}" on this shelf.

Search the shelf carefully and report:
- Found: Yes or No
- How many instances of "{product_name}" found
- Location: shelf level and position for each match
- Facing count
- Stock level
- Similar products nearby (that are NOT the same product)
- Share of shelf: what % this product occupies
- Recommendation for improving this product presence

IMPORTANT: Report ONLY about "{product_name}". Do NOT list all products on the shelf.'''

PROMPTS['planogram'] = '''Image 1 is the REFERENCE PLANOGRAM (how the shelf SHOULD look).
Image 2 is the CURRENT shelf photo.

Compare them and report:
- Compliance score: 0-100%
- Positions checked vs compliant
- Each deviation: what position, what was expected, what is actually there
- Missing products (in reference but not on current shelf)
- Extra products (on shelf but not in reference)
- Top priority fix'''

PROMPTS['competition'] = '''Analyze brand competition on this shelf.

Report:
- Dominant brand (most shelf space)
- Brand ranking: list each brand with share of shelf %, facing count, whether at eye level
- Competitive insights (2-3 observations)
- Threat level: low / medium / high
- Biggest opportunity for improvement'''

PROMPTS['detection_count'] = '''Count every product on this shelf precisely.

For each shelf level (top to bottom):
- List every product with its name, facing count, estimated depth (single/double/deep), estimated total units, and stock status (full/partial/low/empty)

Summary:
- Total unique products
- Total facings
- Total estimated units
- Products needing restock (ordered by urgency)

Use simple names. List each product ONCE per shelf level.'''

print(f'{len(PROMPTS)} modes loaded.')

6 modes loaded.


# Gradio UI - Retail Shelf Intelligence

In [8]:

import gradio as gr

CSS = """
.gradio-container{max-width:1100px!important;margin:0 auto!important;font-family:'Inter',-apple-system,sans-serif!important}
.vk-h{background:linear-gradient(135deg,#0A0F1C,#111827);border-radius:16px;padding:24px 32px;margin-bottom:16px;border:1px solid #1E293B}
.vk-h h1{color:#F8FAFC!important;font-size:28px!important;margin:0 0 4px!important;font-weight:800!important}
.acc{color:#F97316!important}
.vk-h p{color:#94A3B8!important;font-size:12px!important;margin:0!important}
.pills{margin-top:10px;display:flex;gap:5px;flex-wrap:wrap}
.pl{display:inline-block;background:rgba(13,148,136,.1);color:#14B8A6;padding:3px 10px;border-radius:16px;font-size:10px;font-weight:500;border:1px solid rgba(13,148,136,.25)}
.pl-o{background:rgba(249,115,22,.1);color:#F97316;border-color:rgba(249,115,22,.25)}
footer{display:none!important}
"""

MODES = ['Full Shelf Analysis', 'Reference Image Match', 'Search Product by Name',
         'Detection & Count', 'Planogram Compliance', 'Competition Analysis']


def analyze(shelf_img, ref_img, mode, search_text):
    if shelf_img is None:
        return '', '', ''

    # Pick prompt and images
    if mode == 'Reference Image Match':
        if ref_img is None:
            return 'Upload a reference product image first.', '', ''
        raw, lat, tok = run_vlm([ref_img, shelf_img], PROMPTS['reference_match'])

    elif mode == 'Search Product by Name':
        if not search_text or not search_text.strip():
            return 'Enter a product name to search.', '', ''
        prompt = PROMPTS['text_search'].format(product_name=search_text.strip())
        raw, lat, tok = run_vlm(shelf_img, prompt)

    elif mode == 'Planogram Compliance':
        if ref_img is None:
            return 'Upload a reference planogram image first.', '', ''
        raw, lat, tok = run_vlm([ref_img, shelf_img], PROMPTS['planogram'])

    elif mode == 'Competition Analysis':
        raw, lat, tok = run_vlm(shelf_img, PROMPTS['competition'])

    elif mode == 'Detection & Count':
        raw, lat, tok = run_vlm(shelf_img, PROMPTS['detection_count'])

    else:
        raw, lat, tok = run_vlm(shelf_img, PROMPTS['full_analysis'])

    # Stats line
    tps = round(tok / lat, 1) if lat > 0 else 0
    vram = round(torch.cuda.max_memory_allocated() / 1e9, 1) if torch.cuda.is_available() else 0
    stats = f'Model: {MODEL_ID.split("/")[-1]} | Latency: {lat}s | Tokens: {tok} | Tokens/s: {tps} | VRAM: {vram}GB'

    clean = clean_output(raw)
    return clean, stats, raw


with gr.Blocks(css=CSS, title='VK Shelf Intelligence v2', theme=gr.themes.Soft()) as demo:

    gr.HTML(f'''<div class="vk-h">
        <h1>Shelf <span class="acc">Intelligence</span> v2.0</h1>
        <p>Reference matching · Text search · Detection & count · Visibility · Competition · Planogram</p>
        <div class="pills">
            <span class="pl">{MODEL_ID.split('/')[-1]}</span>
            <span class="pl">Reference Matching</span>
            <span class="pl">Text Search</span>
            <span class="pl">Detection & Count</span>
            <span class="pl">Competition</span>
            <span class="pl">Planogram</span>
            <span class="pl">100% Offline</span>
            <span class="pl pl-o">@VK_Venkatkumar</span>
        </div>
    </div>''')

    with gr.Row():
        with gr.Column(scale=2):
            shelf_img = gr.Image(label='Shelf Image', type='pil', height=280)
            ref_img = gr.Image(label='Reference Product (for matching / planogram)', type='pil', height=180)
            mode_radio = gr.Radio(choices=MODES, value='Full Shelf Analysis', label='Analysis Mode')
            search_box = gr.Textbox(label='Product name to search', placeholder='e.g. Calgon, Fairy, Finish...')
            run_btn = gr.Button('Analyze Shelf', variant='primary', size='lg')

        with gr.Column(scale=3):
            stats_text = gr.Textbox(label='Performance', lines=1, interactive=False)
            with gr.Tabs():
                with gr.TabItem('Analysis Result'):
                    result_text = gr.Textbox(label='AI Output', lines=25, show_copy_button=True, interactive=False)
                with gr.TabItem('Copy Raw'):
                    raw_text = gr.Textbox(label='Raw (for integration)', lines=25, show_copy_button=True, interactive=False)

    run_btn.click(fn=analyze, inputs=[shelf_img, ref_img, mode_radio, search_box],
                  outputs=[result_text, stats_text, raw_text])

    gr.HTML('''<div style="text-align:center;padding:14px;color:#64748B;font-size:11px;border-top:1px solid #E2E8F0;margin-top:14px">
        Built by <a href="https://github.com/vk-ant" style="color:#F97316">@VK_Venkatkumar</a>
        · Kaggle Master · M.Tech AI, BITS Pilani · All processing on-device · Your data never leaves this machine.</div>''')

demo.launch(share=True, show_error=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://de86b686599c6be449.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
